# 1. LIBRARIES 

In [14]:
from pathlib import Path
import json

import torch
import pandas as pd

from ultralytics import YOLO

In [27]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Selected device:", DEVICE)

if DEVICE == 0:
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: 0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


# 2. LOAD PRE-TRAINED MODEL YOLO11N-SEG

In [15]:
MODEL_NAME = "yolo11n-seg.pt"
model = YOLO(MODEL_NAME)

model.info()


YOLO11n-seg summary: 203 layers, 2,876,848 parameters, 0 gradients, 10.0 GFLOPs


(203, 2876848, 0, 9.9593344)

# 3. INSPECT MODEL LAYER 

In [16]:
for i, layer in enumerate(model.model.model):
    print(
        f"{i:02d} | "
        f"{layer.__class__.__name__}"
    )

00 | Conv
01 | Conv
02 | C3k2
03 | Conv
04 | C3k2
05 | Conv
06 | C3k2
07 | Conv
08 | C3k2
09 | SPPF
10 | C2PSA
11 | Upsample
12 | Concat
13 | C3k2
14 | Upsample
15 | Concat
16 | C3k2
17 | Conv
18 | Concat
19 | C3k2
20 | Conv
21 | Concat
22 | C3k2
23 | Segment


In [17]:
print("=== MODEL PARAMETER DIAGNOSTIC ===\n")

total = 0
trainable = 0
frozen = 0

for name, param in model.model.named_parameters():
    total += param.numel()

    if param.requires_grad:
        trainable += param.numel()
    else:
        frozen += param.numel()

    print(
        f"{name:70s} | "
        f"requires_grad={param.requires_grad} | "
        f"shape={tuple(param.shape)}"
    )

print("\n=== SUMMARY ===")
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters:    {frozen:,}")

=== MODEL PARAMETER DIAGNOSTIC ===

model.0.conv.weight                                                    | requires_grad=False | shape=(16, 3, 3, 3)
model.0.bn.weight                                                      | requires_grad=False | shape=(16,)
model.0.bn.bias                                                        | requires_grad=False | shape=(16,)
model.1.conv.weight                                                    | requires_grad=False | shape=(32, 16, 3, 3)
model.1.bn.weight                                                      | requires_grad=False | shape=(32,)
model.1.bn.bias                                                        | requires_grad=False | shape=(32,)
model.2.cv1.conv.weight                                                | requires_grad=False | shape=(32, 32, 1, 1)
model.2.cv1.bn.weight                                                  | requires_grad=False | shape=(32,)
model.2.cv1.bn.bias                                                    | requires_

In [20]:

for param in model.model.parameters():
    param.requires_grad = True

print("All parameters restored to trainable.")

All parameters restored to trainable.


In [21]:
trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

frozen_params = sum(
    p.numel()
    for p in model.model.parameters()
    if not p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

Total parameters:     2,876,848
Trainable parameters: 2,876,848
Frozen parameters:    0


# 4. FREEZE BACKBONE LAYER 

In [22]:
BACKBONE_END = 10

for i, layer in enumerate(model.model.model):
    if i <= BACKBONE_END:
        for param in layer.parameters():
            param.requires_grad = False

print("Backbone layers 00–10 frozen.")

Backbone layers 00–10 frozen.


In [23]:
for i, layer in enumerate(model.model.model):
    params = list(layer.parameters())

    if len(params) == 0:
        status = "NO PARAMETERS"
    else:
        trainable = any(p.requires_grad for p in params)
        status = "TRAINABLE" if trainable else "FROZEN"

    print(
        f"{i:02d} | "
        f"{layer.__class__.__name__:12s} | "
        f"{status}"
    )

00 | Conv         | FROZEN
01 | Conv         | FROZEN
02 | C3k2         | FROZEN
03 | Conv         | FROZEN
04 | C3k2         | FROZEN
05 | Conv         | FROZEN
06 | C3k2         | FROZEN
07 | Conv         | FROZEN
08 | C3k2         | FROZEN
09 | SPPF         | FROZEN
10 | C2PSA        | FROZEN
11 | Upsample     | NO PARAMETERS
12 | Concat       | NO PARAMETERS
13 | C3k2         | TRAINABLE
14 | Upsample     | NO PARAMETERS
15 | Concat       | NO PARAMETERS
16 | C3k2         | TRAINABLE
17 | Conv         | TRAINABLE
18 | Concat       | NO PARAMETERS
19 | C3k2         | TRAINABLE
20 | Conv         | TRAINABLE
21 | Concat       | NO PARAMETERS
22 | C3k2         | TRAINABLE
23 | Segment      | TRAINABLE


# 5. STAGE 1 FINE TUNING CONFIGURATION 

In [25]:
PROJECT_ROOT = Path.cwd().parent
DATA_YAML = PROJECT_ROOT / "datasets" / "data.yaml"

In [28]:


STAGE1_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 30,
    "imgsz": 640,
    "batch": 8,
    "device": DEVICE,

    # Optimizer
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "weight_decay": 0.0005,

    # Training behavior
    "warmup_epochs": 3,
    "patience": 10,

    # Augmentation
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 5.0,
    "translate": 0.1,
    "scale": 0.5,
    "fliplr": 0.5,

    # Performance
    "amp": True,

    # Output
    "project": "runs/segment",
    "name": "stage1_yolo11n_seg",
    "exist_ok": True,

    # Reproducibility
    "seed": 42,
}

In [29]:


print("Model:", "YOLO11n-Seg")
print("Dataset:", DATA_YAML)
print("Device:", DEVICE)

print("\nParameters:")

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print(f"Total:      {total_params:,}")
print(f"Trainable:  {trainable_params:,}")
print(f"Frozen:     {frozen_params:,}")

print("\nExpected:")
print("Backbone 00–10 → FROZEN")
print("Neck 11–22     → TRAINABLE")
print("Head 23        → TRAINABLE")

Model: YOLO11n-Seg
Dataset: d:\PREP_INTERN\nutrivision_pro\datasets\data.yaml
Device: 0

Parameters:
Total:      2,876,848
Trainable:  1,511,376
Frozen:     1,365,472

Expected:
Backbone 00–10 → FROZEN
Neck 11–22     → TRAINABLE
Head 23        → TRAINABLE


# 6. FINE TUNING 

In [30]:
stage1_results = model.train(
    data=str(DATA_YAML),

    # Training
    epochs=STAGE1_CONFIG["epochs"],
    imgsz=STAGE1_CONFIG["imgsz"],
    batch=STAGE1_CONFIG["batch"],
    device=STAGE1_CONFIG["device"],

    # Optimizer
    optimizer=STAGE1_CONFIG["optimizer"],
    lr0=STAGE1_CONFIG["lr0"],
    lrf=STAGE1_CONFIG["lrf"],
    weight_decay=STAGE1_CONFIG["weight_decay"],

    # Warmup
    warmup_epochs=STAGE1_CONFIG["warmup_epochs"],

    # Early stopping
    patience=STAGE1_CONFIG["patience"],

    # Augmentation
    hsv_h=STAGE1_CONFIG["hsv_h"],
    hsv_s=STAGE1_CONFIG["hsv_s"],
    hsv_v=STAGE1_CONFIG["hsv_v"],
    degrees=STAGE1_CONFIG["degrees"],
    translate=STAGE1_CONFIG["translate"],
    scale=STAGE1_CONFIG["scale"],
    fliplr=STAGE1_CONFIG["fliplr"],

    # Performance
    amp=STAGE1_CONFIG["amp"],

    # Reproducibility
    seed=STAGE1_CONFIG["seed"],

    # Output
    project=STAGE1_CONFIG["project"],
    name=STAGE1_CONFIG["name"],
    exist_ok=STAGE1_CONFIG["exist_ok"],

    # Visualization / logging
    plots=True,
    verbose=True,
)

Ultralytics 8.4.147  Python-3.13.5 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\PREP_INTERN\nutrivision_pro\datasets\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

: 

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch: 2.14.0+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB


# 7. VALIDATION FINE TUNING STAGE 1 

## 7.1. Load best.pt

In [2]:
from ultralytics import YOLO

MODEL_PATH = r"D:\PREP_INTERN\nutrivision_pro\runs\segment\runs\segment\stage1_yolo11n_seg\weights\best.pt"
DATA_YAML = r"D:\PREP_INTERN\nutrivision_pro\datasets\data.yaml"

model = YOLO(MODEL_PATH)

print("Best model loaded.")

Best model loaded.


In [3]:
results = model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    plots=True,
    verbose=True
)

Ultralytics 8.4.147  Python-3.13.5 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11n-seg summary (fused): 113 layers, 2,837,298 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 282.951.4 MB/s, size: 68.8 KB)
val: Scanning D:\PREP_INTERN\nutrivision_pro\datasets\valid\labels.cache... 37 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 37/37 8.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.6it/s 1.8s.3ss
                   all         37        534      0.573      0.319      0.352       0.27       0.44      0.389      0.348      0.257
                  beef          3          9     0.0749      0.222     0.0391      0.027     0.0505      0.333     0.0397     0.0187
               chicken         17        116      0.511      0.181      0.277      0.205      0.417      0.319      0.284      0.183
  